# Recourse Fairness — baseline setup

Adult + COMPAS, baseline logistic regression, group fairness metrics before any mitigation.

In [ ]:
# pandas<3.0 — aif360 breaks on pandas 3's default string dtype
!pip install "pandas<3.0" -q
!pip install aif360 scikit-learn scipy matplotlib -q
print("installed — restart runtime now")

Restart runtime here (`Runtime` → `Restart session`), then continue. Don't rerun the cell above.

In [ ]:
import aif360, os

RAW = os.path.join(os.path.dirname(aif360.__file__), "data", "raw")

# COMPAS — ProPublica's original release
os.makedirs(f"{RAW}/compas", exist_ok=True)
!wget -q "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv" -O "{RAW}/compas/compas-scores-two-years.csv"

# Adult — raw UCI files, aif360 wants these three exact names
os.makedirs(f"{RAW}/adult", exist_ok=True)
!wget -q "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data" -O "{RAW}/adult/adult.data"
!wget -q "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test" -O "{RAW}/adult/adult.test"
!wget -q "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.names" -O "{RAW}/adult/adult.names"

for f in ["compas/compas-scores-two-years.csv", "adult/adult.data", "adult/adult.test", "adult/adult.names"]:
    p = f"{RAW}/{f}"
    size = os.path.getsize(p) if os.path.exists(p) else 0
    print(f, "OK" if size else "MISSING")

If Adult shows MISSING, UCI's moved the file again — send me whichever filename failed.

In [ ]:
from aif360.datasets import AdultDataset, CompasDataset

adult = AdultDataset()
compas = CompasDataset()

print("Adult:", adult.features.shape, adult.protected_attribute_names)
print("COMPAS:", compas.features.shape, compas.protected_attribute_names)

## Baseline

No mitigation yet — this is the "before" row for Table 1.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from aif360.metrics import ClassificationMetric

def run_baseline(dataset, protected_attr, priv, unpriv, name):
    train, test = dataset.split([0.8], shuffle=True, seed=42)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(train.features)
    X_test = scaler.transform(test.features)

    clf = LogisticRegression(max_iter=2000)
    clf.fit(X_train, train.labels.ravel())

    test_pred = test.copy()
    test_pred.labels = clf.predict(X_test).reshape(-1, 1)

    metric = ClassificationMetric(
        test, test_pred,
        unprivileged_groups=[{protected_attr: unpriv}],
        privileged_groups=[{protected_attr: priv}],
    )

    print(f"--- {name} ({protected_attr}) ---")
    print("SPD:", metric.statistical_parity_difference())
    print("AOD:", metric.average_odds_difference())
    print("Acc:", metric.accuracy())
    print()
    return clf, scaler, test, test_pred, metric

adult_clf, adult_scaler, adult_test, adult_test_pred, adult_metric = run_baseline(
    adult, "sex", priv=1, unpriv=0, name="Adult"
)
compas_clf, compas_scaler, compas_test, compas_test_pred, compas_metric = run_baseline(
    compas, "race", priv=1, unpriv=0, name="COMPAS"
)

Reference run for COMPAS, yours won't match exactly (different split) but should be close:
SPD -0.176, AOD -0.154, Acc 0.672

Next: margin function, then Reweighing.

## Margin (recourse cost)

For every test point the model denies, this is its distance to the decision boundary in
scaled feature space — margin(x) = (w·x + b) / ‖w‖. Bigger distance, harder recourse.
Validated against sklearn's own `decision_function` before trusting any real numbers off it.

One change from the paper draft: computing this for every *unfavorably predicted* instance,
not "correctly classified and unfavorably predicted." Recourse is a property of what the
deployed model asks of you, it doesn't care whether the model's prediction happened to match
the true label — restricting to correct predictions would drop exactly the people (false
negatives) a fairness analysis should care most about. Methods section needs that sentence
fixed to match.

In [ ]:
def recourse_costs(clf, scaler, favorable_label, test_set, test_pred, protected_attr, priv, unpriv):
    X = scaler.transform(test_set.features)
    w = clf.coef_[0]
    b = clf.intercept_[0]
    w_norm = np.linalg.norm(w)

    margin = (X @ w + b) / w_norm
    assert np.allclose(margin, clf.decision_function(X) / w_norm), "margin doesn't match decision_function"

    predicted = test_pred.labels.ravel()
    denied = predicted != favorable_label
    cost = np.abs(margin[denied])

    attr_idx = test_set.protected_attribute_names.index(protected_attr)
    group = test_set.protected_attributes[:, attr_idx][denied]

    return cost[group == priv], cost[group == unpriv]


import numpy as np
from scipy.stats import mannwhitneyu, shapiro

def compare_recourse(priv_cost, unpriv_cost, name):
    print(f"--- {name} recourse cost ---")
    print(f"privileged   n={len(priv_cost)}  mean={priv_cost.mean():.4f}")
    print(f"unprivileged n={len(unpriv_cost)}  mean={unpriv_cost.mean():.4f}")

    # shapiro gets unreliable at large n, subsample if needed
    p_sample = priv_cost[:500]
    u_sample = unpriv_cost[:500]
    print(f"shapiro p (subsample): priv={shapiro(p_sample).pvalue:.4g}  unpriv={shapiro(u_sample).pvalue:.4g}")

    u_stat, p_val = mannwhitneyu(priv_cost, unpriv_cost)
    r = 1 - (2 * u_stat) / (len(priv_cost) * len(unpriv_cost))  # rank-biserial effect size
    print(f"Mann-Whitney U={u_stat:.1f}  p={p_val:.6f}  r={r:.4f}")
    print()

adult_priv_cost, adult_unpriv_cost = recourse_costs(
    adult_clf, adult_scaler, adult.favorable_label, adult_test, adult_test_pred, "sex", 1, 0
)
compas_priv_cost, compas_unpriv_cost = recourse_costs(
    compas_clf, compas_scaler, compas.favorable_label, compas_test, compas_test_pred, "race", 1, 0
)

compare_recourse(adult_priv_cost, adult_unpriv_cost, "Adult")
compare_recourse(compas_priv_cost, compas_unpriv_cost, "COMPAS")

Reference run, COMPAS (won't match exactly, different split, should be close):
n denied 476/1234, privileged mean 0.652, unprivileged mean 0.603, Mann-Whitney p = 0.854
— not significant on the unmitigated baseline. That's a real result, not a bug, worth sitting
with rather than expecting a dramatic gap before any mitigation has even been applied.

Copy all four numbers (both datasets, both groups, both p-values) into Table 2's Baseline row.
Next: Reweighing.

## Reweighing

First mitigation technique, pre-processing: reweights training rows by group/label combo
before the model ever sees them, doesn't touch features or labels, just how much attention
each row gets during fitting. Same margin and stats code from above, reused as-is on the new
model, that's the whole point of building it as functions earlier.

In [ ]:
from aif360.algorithms.preprocessing import Reweighing

def run_reweighing(dataset, protected_attr, priv, unpriv, name):
    train, test = dataset.split([0.8], shuffle=True, seed=42)

    RW = Reweighing(unprivileged_groups=[{protected_attr: unpriv}], privileged_groups=[{protected_attr: priv}])
    train_rw = RW.fit_transform(train)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(train.features)
    X_test = scaler.transform(test.features)

    clf = LogisticRegression(max_iter=2000)
    clf.fit(X_train, train.labels.ravel(), sample_weight=train_rw.instance_weights)

    test_pred = test.copy()
    test_pred.labels = clf.predict(X_test).reshape(-1, 1)

    metric = ClassificationMetric(
        test, test_pred,
        unprivileged_groups=[{protected_attr: unpriv}],
        privileged_groups=[{protected_attr: priv}],
    )
    print(f"--- {name} after Reweighing ---")
    print("SPD:", metric.statistical_parity_difference())
    print("AOD:", metric.average_odds_difference())
    print("Acc:", metric.accuracy())

    return clf, scaler, test, test_pred


adult_rw_clf, adult_rw_scaler, adult_rw_test, adult_rw_pred = run_reweighing(
    adult, "sex", priv=1, unpriv=0, name="Adult"
)
compas_rw_clf, compas_rw_scaler, compas_rw_test, compas_rw_pred = run_reweighing(
    compas, "race", priv=1, unpriv=0, name="COMPAS"
)

adult_rw_priv, adult_rw_unpriv = recourse_costs(
    adult_rw_clf, adult_rw_scaler, adult.favorable_label, adult_rw_test, adult_rw_pred, "sex", 1, 0
)
compas_rw_priv, compas_rw_unpriv = recourse_costs(
    compas_rw_clf, compas_rw_scaler, compas.favorable_label, compas_rw_test, compas_rw_pred, "race", 1, 0
)

compare_recourse(adult_rw_priv, adult_rw_unpriv, "Adult, after Reweighing")
compare_recourse(compas_rw_priv, compas_rw_unpriv, "COMPAS, after Reweighing")

Reference, COMPAS after Reweighing: SPD 0.0015, AOD 0.027, Acc 0.649,
recourse priv mean 0.575, unpriv mean 0.625, Mann-Whitney p=0.154, r=0.081.
Group fairness improved sharply. Recourse gap did not, still not significant, same as baseline.
That's a real result: Reweighing fixed Question 1 without touching Question 2.

Copy all of this into Table 1 and Table 2's Reweighing row for both datasets.
Next: Prejudice Remover, same pattern again.